<div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap;">
    <div style="flex: 1; max-width: 400px; display: flex; justify-content: center;">
        <img src="https://magic.novaims.unl.pt/media/1tdf2arr/ims25_horizontal__positivo_rgb.svg" style="max-width: 70%; height: auto; margin-top: 50px; margin-bottom: 50px;margin-left: 6rem;">
    </div>
    <div style="flex: 2; text-align: center; margin-top: 20px;margin-left: 6rem;">
        <div style="font-size: 28px; font-weight: bold; line-height: 1.2;">
            <span style="color: #FFCD41;">Thesis Project |</span> <span style="color: #F58228;">LLM-Powered Urban Exploration: A Framework for Adaptive Tourist and Mobility Route Planning</span>
        </div>
        <div style="font-size: 17px; font-weight: bold; margin-top: 10px;">
            2025 - 2026
        </div>
        <div style="font-size: 17px; font-weight: bold;">
            Master in Data Science and Advanced Analytics
        </div>
        <div style="margin-top: 20px;">
            <div>André Filipe Gomes Silvestre, 20240502</div>
        </div>
    </div>
</div>

<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: linear-gradient(to right, #F58228, #FFCD41);
            padding: 15px; color: white; border-radius: 300px; text-align: center;">
    <center><h1 style="margin-left: 100px;margin-top: 10px; margin-bottom: 4px; color: white;
                       font-size: 32px; font-family: 'Avenir Next LT Pro', sans-serif;"><b>API Integration: IPMA, Metro, Carris, CP</b></h1></center>
</div>

<br><br>

## **📝 Notebook Overview**

This notebook demonstrates how to interact with various public APIs relevant to the thesis project. It includes examples of API calls, error handling, and data processing for:

1.  **IPMA (Instituto Português do Mar e da Atmosfera)**: Weather forecasts and warnings.
2.  **Metro de Lisboa**: Real-time service status.
3.  **Carris Metropolitana**: Alerts, stops, lines, and routes.
4.  **Comboios de Portugal (CP)**: Real-time train status and station information.

The goal is to establish a robust data collection pipeline for the intelligent tourist agent.

<br><br>

In [ ]:
# Required libraries:
# pip install requests pandas

In [ ]:
import requests
import pandas as pd
import json
import time
from datetime import datetime
from typing import Dict, List, Any, Optional

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## <span style="color: #ffffff;">1. IPMA API (Weather)</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 1. IPMA API (Weather)</b></h2></center>
</div>

<br><br>

The IPMA API provides weather forecasts and warnings. For this project, we focus on the Lisbon area.

**Endpoints:**
- **Warnings:** `https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json`
- **Daily Forecast:** `https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/{globalIdLocal}.json`
    - **Lisbon Global ID:** `1110600`

### **🔎 Dataset Attributes**

<center><b>Table 1 | </b> IPMA API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|
| **1** | `text`                             | Description of the warning (only for yellow/orange/red levels)                  | **`Text`**         |
| **2** | `awarenessTypeName`                | Type of warning (e.g., "Agitação Marítima", "Vento")                            | **`Text`**         |
| **3** | `awarenessLevelID`                 | Warning level color (green, yellow, orange, red)                                | **`Text`**         |
| **4** | `startTime`                        | Start time of the warning                                                       | **`DateTime`**     |
| **5** | `endTime`                          | End time of the warning                                                         | **`DateTime`**     |
| **6** | `precipitaProb`                    | Probability of precipitation (%)                                                | **`Numeric`**      |
| **7** | `tMin`                             | Minimum temperature (°C)                                                        | **`Numeric`**      |
| **8** | `tMax`                             | Maximum temperature (°C)                                                        | **`Numeric`**      |
| **9** | `predWindDir`                      | Predominant wind direction (e.g., N, SW)                                        | **`Text`**         |
| **10**| `forecastDate`                     | Date of the forecast                                                            | **`Date`**         |
</center>

<br><br>

In [7]:
def get_ipma_warnings() -> pd.DataFrame:
    """
    Fetches weather warnings from the IPMA API.

    Returns:
        pd.DataFrame: A DataFrame containing the warnings.
    """
    url = "https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Filter for Lisbon area if needed, but the API returns a list. 
        # Assuming we want to see all warnings or filter later.
        # The prompt mentions "idAreaAviso": "LSB" for Lisbon in the context, but the example response shows "BGC".
        # We will return all and filter for demonstration.
        
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"Error fetching IPMA warnings: {e}")
        return pd.DataFrame()

def get_ipma_forecast(global_id_local: int = 1110600) -> pd.DataFrame:
    """
    Fetches the daily weather forecast for a specific location.

    Args:
        global_id_local (int): The global ID of the location (default is 1110600 for Lisbon).

    Returns:
        pd.DataFrame: A DataFrame containing the daily forecast.
    """
    url = f"https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/{global_id_local}.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'data' in data:
            df = pd.DataFrame(data['data'])
            # Add metadata columns
            df['globalIdLocal'] = data.get('globalIdLocal')
            df['dataUpdate'] = data.get('dataUpdate')
            return df
        else:
            print("No data found in IPMA forecast response.")
            return pd.DataFrame()
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching IPMA forecast: {e}")
        return pd.DataFrame()

# --- Example Usage ---
print("--- IPMA Warnings ---")
df_warnings = get_ipma_warnings()
if not df_warnings.empty:
    # Display first 5 rows
    display(df_warnings.head())
else:
    print("No warnings data available.")

print("\n--- IPMA Forecast (Lisbon) ---")
df_forecast = get_ipma_forecast()
if not df_forecast.empty:
    display(df_forecast.head())
else:
    print("No forecast data available.")

--- IPMA Warnings ---


,text,awarenessTypeName,idAreaAviso,startTime,awarenessLevelID,endTime
0,,Agitação Marítima,BGC,2025-12-10T07:13:00,green,2025-12-13T07:00:00
1,,Nevoeiro,BGC,2025-12-10T07:13:00,green,2025-12-13T07:00:00
2,,Tempo Quente,BGC,2025-12-10T07:13:00,green,2025-12-13T07:00:00
3,,Tempo Frio,BGC,2025-12-10T07:13:00,green,2025-12-13T07:00:00
4,,Precipitação,BGC,2025-12-10T07:13:00,green,2025-12-13T07:00:00



--- IPMA Forecast (Lisbon) ---


,precipitaProb,tMin,tMax,predWindDir,idWeatherType,classWindSpeed,longitude,forecastDate,latitude,classPrecInt,globalIdLocal,dataUpdate
0,22.0,10.6,16.6,N,2,1,-9.1286,2025-12-10,38.7660,NaN,1110600,2025-12-10T12:31:02
1,65.0,8.6,15.4,S,7,1,-9.1286,2025-12-11,38.7660,1.0,1110600,2025-12-10T12:31:02
2,100.0,10.4,12.8,NE,9,1,-9.1286,2025-12-12,38.7660,2.0,1110600,2025-12-10T12:31:02
3,16.0,10.0,17.0,NE,2,1,-9.1286,2025-12-13,38.7660,NaN,1110600,2025-12-10T12:31:02
4,0.0,10.1,16.6,NE,2,1,-9.1286,2025-12-14,38.7660,NaN,1110600,2025-12-10T12:31:02


## <span style="color: #ffffff;">2. Metro de Lisboa API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 2. Metro de Lisboa API</b></h2></center>
</div>

<br><br>

The Metro de Lisboa API provides the current status of the metro lines.

**Endpoint:** `https://app.metrolisboa.pt/status/getLinhas.php`

### **🔎 Dataset Attributes**

<center><b>Table 2 | </b> Metro de Lisboa API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `linha`                            | Name of the metro line (amarela, azul, verde, vermelha)                         | **`Text`**         |
| **2** | `status`                           | Operational status of the line (e.g., "Ok")                                     | **`Text`**         |
| **3** | `tipo_msg`                         | Message type code (internal use)                                                | **`Text`**         |

</center>

<br><br>

In [8]:
def get_metro_status() -> pd.DataFrame:
    """
    Fetches the status of Lisbon Metro lines.

    Returns:
        pd.DataFrame: A DataFrame containing the status of each line.
    """
    url = "https://app.metrolisboa.pt/status/getLinhas.php"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'resposta' in data:
            # The API returns a dictionary with line names as keys and status as values.
            # We need to transform this into a more usable format.
            status_data = data['resposta']
            
            # Extract lines and statuses
            lines = ['amarela', 'azul', 'verde', 'vermelha']
            formatted_data = []
            
            for line in lines:
                formatted_data.append({
                    'linha': line,
                    'status': status_data.get(line, 'Unknown'),
                    'tipo_msg': status_data.get(f'tipo_msg_{line[:2]}', '0') # e.g., tipo_msg_am
                })
            
            return pd.DataFrame(formatted_data)
        else:
            print("Unexpected format in Metro API response.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching Metro status: {e}")
        return pd.DataFrame()

# --- Example Usage ---
print("--- Metro de Lisboa Status ---")
df_metro = get_metro_status()
if not df_metro.empty:
    display(df_metro)
else:
    print("No metro data available.")

--- Metro de Lisboa Status ---


,linha,status,tipo_msg
0,amarela,Ok,0
1,azul,Ok,0
2,verde,Ok,0
3,vermelha,Ok,0


## <span style="color: #ffffff;">3. Carris Metropolitana API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 3. Carris Metropolitana API</b></h2></center>
</div>

<br><br>

Carris Metropolitana provides extensive data on bus services in the Lisbon Metropolitan Area.

**Endpoints:**
- **Alerts:** `https://api.carrismetropolitana.pt/v2/alerts`
- **Stops:** `https://api.carrismetropolitana.pt/v2/stops`
- **Lines:** `https://api.carrismetropolitana.pt/v2/lines`
- **Routes:** `https://api.carrismetropolitana.pt/v2/routes`

### **🔎 Dataset Attributes**

<center><b>Table 3 | </b> Carris Metropolitana API Attributes (Selected). <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `alert_id`                         | Unique identifier for the alert                                                 | **`Text`**     |
| **2** | `cause`                            | Cause of the alert (e.g., "DEMONSTRATION")                                      | **`Text`**     |
| **3** | `description_text`                 | Detailed description of the alert                                               | **`Text`**     |
| **4** | `id` (Stop/Line)                   | Unique identifier for the stop or line                                          | **`Text`**     |
| **5** | `long_name`                        | Full name of the stop or line                                                   | **`Text`**     |
| **6** | `lat` / `lon`                      | Geographical coordinates                                                        | **`Numeric`**    |
| **7** | `municipalities`                   | List of municipalities served                                                   | **`List`**         |

</center>

<br><br>

In [9]:
def get_carris_data(endpoint: str) -> pd.DataFrame:
    """
    Generic function to fetch data from Carris Metropolitana API.

    Args:
        endpoint (str): The API endpoint (e.g., 'alerts', 'stops', 'lines').

    Returns:
        pd.DataFrame: A DataFrame containing the requested data.
    """
    url = f"https://api.carrismetropolitana.pt/v2/{endpoint}"
    try:
        response = requests.get(url, timeout=20) # Increased timeout for large datasets like stops
        response.raise_for_status()
        data = response.json()
        
        # The API usually returns a list of objects directly
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"Error fetching Carris {endpoint}: {e}")
        return pd.DataFrame()

# --- Example Usage ---

# 1. Alerts
print("--- Carris Alerts ---")
df_carris_alerts = get_carris_data('alerts')
if not df_carris_alerts.empty:
    # Display relevant columns for alerts
    display(df_carris_alerts[['alert_id', 'cause', 'effect']].head())
else:
    print("No alerts available.")

# 2. Lines (Example)
print("\n--- Carris Lines (First 5) ---")
df_carris_lines = get_carris_data('lines')
if not df_carris_lines.empty:
    display(df_carris_lines[['id', 'short_name', 'long_name']].head())
else:
    print("No lines data available.")

--- Carris Alerts ---


,alert_id,cause,effect
0,8CIRU,VEHICLE_ISSUE,SIGNIFICANT_DELAYS
1,YUZAV,ROAD_INCIDENT,DETOUR
2,S25YN,VEHICLE_ISSUE,SIGNIFICANT_DELAYS
3,MWAYC,DEMONSTRATION,DETOUR
4,PZASB,DEMONSTRATION,DETOUR



--- Carris Lines (First 5) ---


,id,short_name,long_name
0,1001,1001,Alfragide (Estr Seminario) - Reboleira (Estação)
1,1002,1002,Reboleira (Estação) | Circular via Alfragide (...
2,1003,1003,Amadora (Estação Norte) - Amadora Este (Metro)
3,1004,1004,Amadora (Estação Norte) | Circular via Moinhos...
4,1005,1005,Amadora (Estação Norte) - UBBO


## <span style="color: #ffffff;">4. Comboios de Portugal (CP) API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 4. Comboios de Portugal (CP) API</b></h2></center>
</div>

<br><br>

The CP API (via comboios.live) provides real-time information about trains and stations.

**Endpoints:**
- **Stations:** `https://comboios.live/api/stations`
- **Vehicles (Real-time):** `https://comboios.live/api/vehicles`

### **🔎 Dataset Attributes**

<center><b>Table 4 | </b> CP API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `trainNumber`                      | Unique identifier for the train                                                 | **`Numeric`**      |
| **2** | `runDate`                          | Date of the train run                                                           | **`Date`**         |
| **3** | `delay`                            | Current delay in seconds                                                        | **`Numeric`**      |
| **4** | `status`                           | Status of the train (e.g., "IN_TRANSIT")                                        | **`Text`**         |
| **5** | `latitude` / `longitude`           | Current geographical position of the train                                      | **`Numeric`**    |
| **6** | `designation` (Station)            | Name of the station                                                             | **`Text`**         |
| **7** | `code` (Station)                   | Station code                                                                    | **`Text`**         |
</center>

</center>

<br><br>

In [10]:
def get_cp_stations() -> pd.DataFrame:
    """
    Fetches the list of CP stations.

    Returns:
        pd.DataFrame: A DataFrame containing station information.
    """
    url = "https://comboios.live/api/stations"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'stations' in data:
            df = pd.DataFrame(data['stations'])
            return df
        else:
            print("Unexpected format in CP Stations API response.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching CP stations: {e}")
        return pd.DataFrame()

def get_cp_vehicles() -> pd.DataFrame:
    """
    Fetches real-time information about CP vehicles (trains).

    Returns:
        pd.DataFrame: A DataFrame containing vehicle information.
    """
    url = "https://comboios.live/api/vehicles"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'vehicles' in data:
            df = pd.DataFrame(data['vehicles'])
            return df
        else:
            print("Unexpected format in CP Vehicles API response.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching CP vehicles: {e}")
        return pd.DataFrame()

# --- Example Usage ---
print("--- CP Stations (First 5) ---")
df_cp_stations = get_cp_stations()
if not df_cp_stations.empty:
    display(df_cp_stations.head())
else:
    print("No station data available.")

print("\n--- CP Vehicles (Real-time) ---")
df_cp_vehicles = get_cp_vehicles()
if not df_cp_vehicles.empty:
    # Display relevant columns
    display(df_cp_vehicles[['trainNumber', 'status', 'delay', 'latitude', 'longitude']].head())
else:
    print("No vehicle data available.")

--- CP Stations (First 5) ---


,code,designation,latitude,longitude,region,railways
0,94-52001,Abrantes,39.4401931762695,-8.19432544708252,None,[]
1,94-36046,Ademia,40.2509269714355,-8.45034217834473,None,[]
2,94-18119,Afife,41.7753791809082,-8.86147117614746,None,[1]
3,94-61002,Agualva - Cacem,38.7664260864258,-9.29842472076416,None,[]
4,94-3046,Aguas Santas - Palmilheira,41.2017288208008,-8.55673980712891,None,[1]



--- CP Vehicles (Real-time) ---


,trainNumber,status,delay,latitude,longitude
0,122,IN_TRANSIT,955.0,40.1697331,-8.6254033
1,123,CANCELLED,NaN,None,None
2,125,IN_TRANSIT,-82.0,39.75478120000532,-8.563088199718912
3,132,AT_STATION,0.0,41.455265000159905,-8.5443301654626
4,421,CANCELLED,NaN,None,None


<div style="background: linear-gradient(to right, #F58228, #FFCD41);
            padding: 3px; color: white; border-radius: 900px; text-align: center;">
</div>

<br>

## **🏁 Conclusion**

1.  **IPMA**: Weather data is crucial for suggesting outdoor vs. indoor activities.
2.  **Metro de Lisboa**: Service status helps in avoiding delays.
3.  **Carris Metropolitana**: Comprehensive bus network data allows for detailed route planning.
4.  **CP**: Real-time train data supports regional mobility.

The functions defined here can be modularized and integrated into the main agent's codebase to provide real-time context for itinerary planning.